# Collect through the existing Chrome session

Version **0.2.17** introduced `collection_mode="session"`: queued component GETs run sequentially in the existing Sofascore tab using its recently successful site-issued session headers. Only missing components are requested, bodies are acknowledged directly by Python, and session values stay in Chrome. It does not navigate through H2H or open each queued match. The first live metadata test recovered HTTP 200 for blocked match **12448325** in 240 ms on 13 September; sustained collection is still being measured. The original passive collector remains available as `collection_mode="page"`. Python still sends no HTTP requests. No VPN is used.

This notebook prepares and monitors `chrome-from-queue-bc6ae7b3`: **40,060 matches across 109 validated seasons** from the other scraper's metadata. `Eredivisie_16_17` remains unresolved and excluded. The earlier pilot `chrome-pilot-16d25576` is still available.

Run cells in order. Defaults inspect the saved queue; they do not start, recreate, or pause collection. Queue collection and complete-season publication remain separate.
Version **0.2.20** runs inter-request pacing through the existing Python connection with a high-resolution monotonic clock. Minimum spacing and optional random delay retain their configured meaning. Pause controls, memory renewals, and challenge/season rests remain enabled. Transient Windows sharing conflicts during status-file replacement receive bounded retries. Reports retain pacing backend and requested/actual wait totals.


In [ ]:
from pathlib import Path
import sys, json
from dataclasses import replace

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scraping/sofascore_web_v3").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the xDiyo project")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scraping.sofascore_web_v3.capture import PROFILE
from scraping.sofascore_web_v3.campaign import BASE, prepare_campaign, status, request_pause, update_policy, validate_policy
print("Project:", ROOT)
print("Python:", sys.executable)

## Configuration

`QUEUE_SOURCE = "existing_scraper"` reuses `data/xDiyo_data/_collection/inventory.json`. It preserves season order and takes match IDs from `legacy_event_ids`; source CSVs provide only routing/identity columns. Existing provider slugs are used when cached, otherwise the saved team slugs and matchup ID construct the match route. New browser metadata must confirm the fixture before exporting results.

No additional fixture-discovery requests are needed for the valid queued IDs. A missing or invalid queued season remains a reported gap. Scores, standings, shots, lineups, heatmaps and flags from the legacy files are not copied into the new collection.

Set `SOURCE_SEASON_STEMS` to a list of existing season names or leave it `None` for all valid seasons. `SOURCE_MAX_MATCHES` optionally bounds a new queue. `QUEUE_SOURCE = "manual"` retains the explicit `EVENT_IDS` option.

Set `COLLECTION_MODE = "session"` and `SESSION_REQUEST_INTERVAL_SECONDS = 0.5` in the notebook. The interval is a minimum between request starts, across match boundaries. `RANDOM_ACTION_DELAY_MAX_SECONDS = None` disables jitter; a value T adds a random [0,T] wait before each component request in session mode. Five-minute season rests remain. The saved queue configuration is authoritative: disconnect the writer and apply the policy once with `APPLY_SAVED_POLICY = True`. This changes no fixture IDs, feature flags, captured data, or completed results.

`MAX_API_OBSERVATIONS_PER_RUN = None` disables the cumulative observation cap. A positive integer optionally limits observations in one Python connection. The per-match limit remains separate. Observation budgets pause future extension actions; they are not an hourly API quota and cannot stop the website's own background traffic. The default five-minute break applies when the queued season changes.

`VIEW_WAIT_SECONDS` is a maximum observation window for page/section changes. The collector continues early once the expected data is saved and controls are ready. A missing response gets the full window. The per-match observation cap counts selected data; other API traffic remains diagnostic metadata, with a separate 5,000-entry recorder guard. Neither control changes the website's own request volume.

Session mode keeps an ordinary Sofascore match page as its session anchor. It renews that same page between saved matches when the configured memory interval is reached, or when the session has insufficient validity for a match. The request loop never uses H2H. A renewal that needs interaction pauses. Optional bounded recovery for a 403 challenge is described above; other denials pause immediately. Python validates captured metadata against the queued event, teams, competition and season, then requires all selected endpoints to be accounted for before completing the match. The report records `collection_mode="session"`, `identity_source="captured_event_metadata"`, the explicit request count, and renewal evidence. 404 means unavailable data; it never fabricates a populated feature. Pregame positions remain distinct from H2H data.

With `SESSION_CHALLENGE_RECOVERY = True`, one HTTP 403 whose reason is exactly `challenge` gets a configurable rest (`SESSION_CHALLENGE_COOLDOWN_SECONDS = 180`), one ordinary renewal of the existing anchor, and one retry of the failed component. Previously saved components are reused. The first denial is retained, and the entire match has only one recovery attempt. A second denial, 401, 429, another 403 reason, failed renewal, or visible human challenge pauses. Set the flag to `False` for immediate pauses on all denials. Session values stay inside Chrome, and neither H2H nor a VPN is used.

Renewal requires a successful observation that began after the page was renewed; old Network entries cannot validate it. If a token is nearly expired, the collector waits until it expires before reloading, because the site's initializer otherwise keeps the same token. These fixed rests check local pause controls without component requests and are added to the match deadline; renewal and request work still have the normal time budget. Reports retain request counts, renewal counts, recovery attempts, and wait durations.


In [ ]:
EXTENSION_ID = "bhkjpmoblllmoheemejdcemnhmdcdnbg"
CAMPAIGN_ID = "chrome-from-queue-bc6ae7b3"  # Keep this value to resume the same queue.
QUEUE_SOURCE = "existing_scraper"  # "existing_scraper" or "manual"
SOURCE_INVENTORY = ROOT / "data/xDiyo_data/_collection/inventory.json"
SOURCE_SEASON_STEMS = None          # e.g. ["Bundesliga_2_24_25"]
SOURCE_MAX_MATCHES = None           # e.g. 3 for a small test
EVENT_IDS = [12499375, 12499386, 12499406]  # Used only by the manual source

INCLUDE_XG = True       # Shotmap, including available xG/xGOT; statistics also retain team xG.
INCLUDE_STATISTICS = True
INCLUDE_HEATMAPS = True
INCLUDE_LINEUPS = True
INCLUDE_INJURIES = True # Reported absences; unknown provider codes stay unclassified.
INCLUDE_GOAL_ACTIONS = True
INCLUDE_PREGAME = True  # Historical match-specific positions, when supplied.
INCLUDE_COMMENTARY = False
DURATION_MODE = "incidents"  # "off", "metadata", or "incidents"; exact duration may be unavailable.

COLLECTION_MODE = "session"  # "session" for component GETs; "page" for passive website navigation.
SESSION_REQUEST_INTERVAL_SECONDS = 0.5  # Minimum spacing between explicit request starts.
SESSION_CHALLENGE_RECOVERY = True  # One recovery attempt per match for reason=challenge.
SESSION_CHALLENGE_COOLDOWN_SECONDS = 180
RANDOM_ACTION_DELAY_MAX_SECONDS = None
BETWEEN_SEASONS_SECONDS = 300
MATCH_TIMEOUT_SECONDS = 90
VIEW_WAIT_SECONDS = 8  # Maximum page/section wait; ends early when data and controls are ready.
MAX_API_OBSERVATIONS_PER_MATCH = 120  # Selected observations; unrelated traffic does not consume this cap.
MAX_API_OBSERVATIONS_PER_RUN = None  # No cumulative observation cap; optional positive integer.
PAUSE_ON_MISSING_CORE = True
MEMORY_RESET_EVERY_MATCHES = 50  # Positive integer, or None to disable periodic renewal.

CREATE_NEW_CAMPAIGN = False
APPLY_SAVED_POLICY = False  # Close DevTools first; True explicitly updates this queue's policy.
PREPARE_NATIVE_HOST = False
REGISTER_NATIVE_HOST = False
REQUEST_PAUSE = False
PUBLISH_COMPLETED_SEASONS = False  # Offline export; safe to run while Chrome collects.
SEASON_EXPORT_STEMS = None  # None exports all complete cohorts; or give an explicit list.

profile = replace(PROFILE, include_shotmap=INCLUDE_XG,
    include_statistics=INCLUDE_STATISTICS, include_heatmaps=INCLUDE_HEATMAPS,
    include_lineups=INCLUDE_LINEUPS, include_injuries=INCLUDE_INJURIES,
    include_goal_actions=INCLUDE_GOAL_ACTIONS, include_pregame=INCLUDE_PREGAME,
    include_comments=INCLUDE_COMMENTARY, duration_mode=DURATION_MODE)
policy = dict(session_challenge_recovery=SESSION_CHALLENGE_RECOVERY,
    session_challenge_cooldown_seconds=SESSION_CHALLENGE_COOLDOWN_SECONDS, collection_mode=COLLECTION_MODE, session_request_interval_seconds=SESSION_REQUEST_INTERVAL_SECONDS,
    random_action_delay_max_seconds=RANDOM_ACTION_DELAY_MAX_SECONDS,
    between_seasons_seconds=BETWEEN_SEASONS_SECONDS, match_timeout_seconds=MATCH_TIMEOUT_SECONDS,
    view_wait_seconds=VIEW_WAIT_SECONDS, max_api_observations_per_match=MAX_API_OBSERVATIONS_PER_MATCH,
    max_api_observations_per_run=MAX_API_OBSERVATIONS_PER_RUN, pause_on_missing_core=PAUSE_ON_MISSING_CORE,
    memory_reset_every_matches=MEMORY_RESET_EVERY_MATCHES)

## Discover 2025/26 or another league

Extension **0.3.0** adds discovery using the existing Chrome tab. The prepared ID below selects all 13 registered leagues and the **regular league stage only**, with xG, heatmaps, lineups and no commentary. Named playoffs and post-split groups are excluded; some older Belgian/Scottish files included those groups.

Leave `CREATE_NEW_DISCOVERY = False` to resume. Saved settings are authoritative. A new league needs a unique filename name, Sofascore competition ID (the final number in its league URL), and its regular-season meetings per team pair. Single-table formats with 1–4 meetings per pair are supported; other formats need a validation rule.

Reload the extension, reopen DevTools, and disconnect the old collection campaign. Paste the discovery ID into **Discover a season**, choose **Create / reconnect**, then **Discover / resume**. The same tab is reused. The random delay maximum below adds a uniform [0,T] wait; `None` disables it.

After all pages are saved and counts reconcile, select **Create collection campaign** in the extension. This connects the new campaign but does not start it. Then choose **Start / resume**. Rerun the preview cell below to pick up its new campaign ID before using the later monitoring/export cells. No match component is fetched by this notebook.

In [ ]:
from scraping.sofascore_web_v3.discovery import DISCOVERY_BASE, Discovery, league_catalog, prepare_discovery

DISCOVERY_ID = "discover-2025-2026-eff840b6"
DISCOVERY_SEASON = "25/26"
DISCOVERY_LEAGUES = league_catalog()  # Select a subset or append a new league definition.
# Example shape: {"name": "League_Name", "competition_id": <provider ID>, "meetings_per_pair": 2}
CREATE_NEW_DISCOVERY = False

if CREATE_NEW_DISCOVERY:
    created_discovery = prepare_discovery(season=DISCOVERY_SEASON, leagues=DISCOVERY_LEAGUES,
        collection_profile=profile, collection_policy=policy)
    DISCOVERY_ID = created_discovery["discovery_id"]
    print("New discovery ID:", DISCOVERY_ID)
else:
    print("Resume discovery:", DISCOVERY_ID)
print("Selected configuration for a NEW discovery:", DISCOVERY_SEASON, "|", len(DISCOVERY_LEAGUES), "leagues")
print("Extra random delay maximum:", RANDOM_ACTION_DELAY_MAX_SECONDS)

In [ ]:
if DISCOVERY_ID:
    with Discovery(DISCOVERY_BASE / DISCOVERY_ID, read_only=True) as discovery:
        discovery_snapshot = discovery.status()
    print(json.dumps(discovery_snapshot, indent=2))
    if discovery_snapshot["campaign"]:
        CAMPAIGN_ID = discovery_snapshot["campaign"]["campaign_id"]
        CREATE_NEW_CAMPAIGN = False
        APPLY_SAVED_POLICY = False
        print("Using discovered collection campaign:", CAMPAIGN_ID)
    else:
        print("No new collection campaign yet. Run discovery in Chrome; the existing campaign ID below is unchanged.")

## One-time local writer setup

Already prepared and registered on this computer for the extension ID above. Leave both flags false unless reinstalling. Preparation builds the small native launcher inside `data/website_capture_v3/native_host`; registration sets only the current user's `Software\Google\Chrome\NativeMessagingHosts\com.xdiyo.sofascore` entry. It requires no local HTTP server and accepts only this extension origin.

Use this project's `.venv` for collection. Registration may need to be run from your own terminal if a managed environment blocks registry writes.

In [ ]:
from scraping.sofascore_web_v3.install_native_host import prepare, register
if PREPARE_NATIVE_HOST:
    print(json.dumps(prepare(EXTENSION_ID), indent=2))
if REGISTER_NATIVE_HOST:
    print(json.dumps(register(), indent=2))
if not (PREPARE_NATIVE_HOST or REGISTER_NATIVE_HOST):
    print("Using the already prepared local writer; setup was not changed.")

## Prepare or resume

To resume, leave `CREATE_NEW_CAMPAIGN = False` and keep the same campaign ID. The saved configuration is authoritative; changing flags above does not silently change a prepared queue.

To prepare a different selection, set `CREATE_NEW_CAMPAIGN = True` once and run the next cell. Save the returned campaign ID, then set the flag back to false. Importing the existing queue checks the source file hashes, season labels, event IDs, and any cached identity against the inventory. It reads the other collector's metadata without changing its queue, stop markers, cache, or outputs.

Preparing the full current inventory read about 1.8 GB of local source CSVs and took about 25 seconds on this computer. Preview and resume only read local metadata. No browser is started by this cell.

The prepared full queue also has its cumulative observation limit disabled. To apply a Python or saved-policy update to an existing connection, close/reopen DevTools, connect the same campaign ID, and select **Start / resume**. Acknowledged responses and completed matches are retained.

To apply the pacing and memory-reset settings to this existing queue: close DevTools to disconnect its writer, set `APPLY_SAVED_POLICY = True`, and run the next cell once. Then set it back to `False`, reload the updated extension, reopen DevTools, and connect/start the same campaign ID. The helper takes the collector lock, backs up the configuration, and changes only policy settings. It preserves fixtures, profiles, observations, and completed results.


In [ ]:
from scraping.sofascore_web_v3.queue_source import queue_preview, prepare_from_scraper_queue
if QUEUE_SOURCE not in {"existing_scraper", "manual"}:
    raise ValueError("QUEUE_SOURCE must be existing_scraper or manual")
if QUEUE_SOURCE == "existing_scraper":
    print("Source queue preview:", json.dumps(queue_preview(SOURCE_INVENTORY,
        season_stems=SOURCE_SEASON_STEMS, max_matches=SOURCE_MAX_MATCHES), indent=2))
if CREATE_NEW_CAMPAIGN:
    if QUEUE_SOURCE == "existing_scraper":
        created = prepare_from_scraper_queue(SOURCE_INVENTORY,
            season_stems=SOURCE_SEASON_STEMS, max_matches=SOURCE_MAX_MATCHES,
            profile=profile, policy=policy)
    else:
        created = prepare_campaign(EVENT_IDS, name="chrome-manual", profile=profile, policy=policy)
    CAMPAIGN_ID = created["campaign_id"]
    print(json.dumps({k: v for k, v in created.items() if k != "event_ids"}, indent=2))

if APPLY_SAVED_POLICY:
    print("Updated saved policy:", update_policy(CAMPAIGN_ID, policy))

campaign_dir = BASE / CAMPAIGN_ID
config = json.loads((campaign_dir / "config.json").read_text(encoding="utf-8"))
print("Campaign:", CAMPAIGN_ID)
print("Directory:", campaign_dir)
print("Total fixtures:", len(config["items"]))
print("Saved policy:", json.dumps(config["policy"], indent=2))
for item in config["items"][:10]:
    print(item["event_id"], item["label"], item["season_year"])
if len(config["items"]) > 10:
    print("Only the first 10 fixture identities are displayed.")
if config["profile"] != vars(profile) or validate_policy(config["policy"]) != validate_policy(policy):
    print("NOTE: parameters above differ from this saved queue. Its saved configuration will be used.")

## Start / resume in Chrome

1. Reload **xDiyo Network Capture** after an update and close/reopen DevTools. Check **Version 0.3.0**.
2. Keep an ordinary Sofascore match page open in the collection tab. Any match is suitable; the queue supplies the IDs. Complete any visible challenge.
3. Enter the saved campaign ID, select **Connect to Python**, and check **Python 0.2.21 connected**. Select **Start / resume**.

Session mode keeps an ordinary Sofascore match page as its session anchor. It renews that same page between saved matches when the configured memory interval is reached, or when the session has insufficient validity for a match. The request loop never uses H2H. A renewal that needs interaction pauses. Optional bounded recovery for a 403 challenge is described above; other denials pause immediately. Python validates captured metadata against the queued event, teams, competition and season, then requires all selected endpoints to be accounted for before completing the match. The report records `collection_mode="session"`, `identity_source="captured_event_metadata"`, the explicit request count, and renewal evidence. 404 means unavailable data; it never fabricates a populated feature. Pregame positions remain distinct from H2H data.

The separate **Session access test** makes one metadata GET without updating the queue. **Export test result** saves only redacted status and identity evidence.

In `page` mode the existing website navigation, optional section loading, and bounded historical-link fallback remain available. Session mode bypasses that navigation entirely. After a pause, completed matches and acknowledged terminal components are skipped.

With `SESSION_CHALLENGE_RECOVERY = True`, one HTTP 403 whose reason is exactly `challenge` gets a configurable rest (`SESSION_CHALLENGE_COOLDOWN_SECONDS = 180`), one ordinary renewal of the existing anchor, and one retry of the failed component. Previously saved components are reused. The first denial is retained, and the entire match has only one recovery attempt. A second denial, 401, 429, another 403 reason, failed renewal, or visible human challenge pauses. Set the flag to `False` for immediate pauses on all denials. Session values stay inside Chrome, and neither H2H nor a VPN is used.

Renewal requires a successful observation that began after the page was renewed; old Network entries cannot validate it. If a token is nearly expired, the collector waits until it expires before reloading, because the site's initializer otherwise keeps the same token. These fixed rests check local pause controls without component requests and are added to the match deadline; renewal and request work still have the normal time budget. Reports retain request counts, renewal counts, recovery attempts, and wait durations.


In [ ]:
snapshot = status(CAMPAIGN_ID)
print("State:", snapshot["state"], "| Counts:", snapshot["counts"])
print("API observations in this connection:", snapshot["api_observations_this_connection"])
for match in snapshot["matches"]:
    print(match["event_id"], match["state"], match["reason"] or "", match["report"] or "")
# Re-run this cell to monitor. It reads local JSON; it does not poll Sofascore.
if snapshot.get("matches_truncated"):
    print("Status lists at most 100 active/recent fixtures; counts cover the entire queue.")

## Inspect captured results

Every completed/partial attempt has a HAR archive, isolated raw-response cache, typed Parquet manifest, endpoint coverage report, and `feature_coverage.json`. These files live below this campaign directory and do not overwrite the legacy restoration or season exports.

The real capture validated so far supplied 39 shots with xG, both team heatmaps, 40 distinct lineup players, three reported absences, and five goal-action sequences. Exact match duration and explicit pass recipients were absent. Historical pregame positions were unavailable, so normalized standings cannot be asserted. Promotion/relegation flags remain a separate season-membership processing step.

In [ ]:
from scraping.sofascore.export import read_tables
snapshot = status(CAMPAIGN_ID)
reports = [Path(m["report"]) for m in snapshot["matches"] if m["report"]]
if not reports:
    print("No match import has completed in this campaign yet.")
else:
    report_path = reports[-1]
    report = json.loads(report_path.read_text(encoding="utf-8"))
    feature_report = json.loads(report_path.with_name("feature_coverage.json").read_text(encoding="utf-8"))
    tables = read_tables(report["manifest"])  # Verifies the Parquet file hashes.
    print("Latest event:", report["event_id"], "| Endpoint coverage complete:", report["complete"])
    print("Rows:", {name: len(rows) for name, rows in tables.items()})
    for name, value in feature_report["features"].items():
        print(name, "->", value["status"])
    for warning in feature_report["warnings"]:
        print("Source note:", warning)

## Export completed seasons

Set `PUBLISH_COMPLETED_SEASONS = True` and run the cell below to assemble complete captured cohorts into `data/xDiyo_data/<original-season-stem>.parquet`. Set `SEASON_EXPORT_STEMS` to a list to choose seasons, or leave it `None` to select all complete cohorts. Restore the publication flag to `False` afterward.

This is offline and can run while Chrome collects another season. It checks source hashes, fixture identities, selected-component accounting, and every resulting table. Incomplete or deferred cohorts are refused. Identical existing exports are verified and reused; conflicting files are preserved and reported. Raw captures stay in their original directories, with source lineage recorded alongside the season files.

The season Parquet keeps one match per row and typed component lists, with a companion relational manifest. Raw historical pregame positions stay raw; the existing compatibility loader computes normalized standings over the assembled season. Missing positions remain missing. Promotion/relegation flags remain separate season-membership processing.


In [ ]:
if PUBLISH_COMPLETED_SEASONS:
    from scraping.sofascore_web_v3.publish import completed_season_stems, publish_captured_season
    capture_directory = ROOT / "data/website_capture_v3/campaigns" / CAMPAIGN_ID
    export_stems = completed_season_stems(capture_directory) if SEASON_EXPORT_STEMS is None else list(SEASON_EXPORT_STEMS)
    if not export_stems:
        print("No complete captured season cohorts are ready.")
    for stem in export_stems:
        result = publish_captured_season(capture_directory, stem, output=ROOT / "data/xDiyo_data")
        print(stem, result["publication_state"], "|", result["matches"], "matches")
else:
    print("Season publication disabled. Set PUBLISH_COMPLETED_SEASONS = True to export complete cohorts.")


## Pause

Use **Pause** in the xDiyo panel, or set `REQUEST_PAUSE = True` and run the cell below. The writer stores a pause marker, checked during the next local progress update. The website itself may continue background requests while left open.

Closing DevTools disconnects the writer and pauses an active match; acknowledged responses survive in SQLite. To resume, reopen DevTools, connect the same campaign ID, and select **Start / resume**. Completed matches are skipped; interrupted matches keep their stored observations. After you complete a site challenge, Start / resume handles returning to the queued match.

In [ ]:
if REQUEST_PAUSE:
    request_pause(CAMPAIGN_ID)
    print("Pause requested for", CAMPAIGN_ID)
else:
    print("No pause requested.")

## Automatically convert the newly collected seasons

Set `WATCH_DISCOVERED_PUBLICATIONS = True` after the extension creates the new campaign. This starts the existing converter in this notebook kernel and stays attached until that campaign finishes or you interrupt the cell. Keep the kernel running. It verifies and skips already published seasons; the collector continues independently. This cell obtains the campaign ID from the discovery record so it cannot accidentally launch a watcher for the old historical campaign.

In [ ]:
WATCH_DISCOVERED_PUBLICATIONS = False
if WATCH_DISCOVERED_PUBLICATIONS:
    from scraping.sofascore_web_v3.publish_watch import watch_publications
    with Discovery(DISCOVERY_BASE / DISCOVERY_ID, read_only=True) as discovery:
        discovered_campaign = discovery.status()["campaign"]
    if not discovered_campaign:
        raise RuntimeError("Create the collection campaign in the extension first")
    print("Watching", discovered_campaign["campaign_id"], flush=True)
    watch_publications(Path(discovered_campaign["directory"]), output=ROOT / "data/xDiyo_data", watch=True)
else:
    print("Discovered-season publication watcher is disabled.")